# Feature Engineering Intro I
<hr style="border:2px solid black">

## Introduction

What is our __goal__ when we __train a machine learning model__ (ML)?

<img src = "./assets/basic_model.png" width=400>

The primary goal when training a machine learning model is to develop a system that can accurately make predictions or decisions based on input data. 
But how do we improve the accuracy of our predictions?

**Selecting the right model** is essential, but it's only the beginning. Beyond model selection, we must explore other strategies to refine our model's learning, resulting in more **accurate predictions**.

Consider the learning process of the model:

When provided with `X_train` input, the algorithm learns parameters to align the model’s output with the known `y_train` values.

To improve predictions, we can also work on how the input data is represented.

So, what strategies can we employ to achieve this?

The goal is to generalize to new data, not simply fit `X_train` and `y_train`. Compare candidate feature transformations on validation data; adding more features does not automatically improve predictions.

## What is Feature Engineering?
The diagram below is **over-simplified**. Raw data may contain missing values, unsuitable representations, or irrelevant inputs that make learning harder. Instead, we need to **clean up** the data and make **creative decisions** to select the **key features** that will enable the model to accurately predict outcomes.

<img src="./assets/feature_eng.png" width="500" alt="Raw data is transformed into features before model training">

Some aspects of **feature engineering** are methodical and consistent. We'll explore these first.

The other aspects are more similar to an **art form**, requiring a deep understanding of the subject and a bit of **human intuition**.

The process of **feature engineering** can be one of the most **time-consuming** parts of modeling, but it's essential. The necessary transformations depend on the data and the model: for example, trees generally do not require numerical scaling, while distance-based models often benefit from it.


**Estimated time spent with data organizing**

<img src="./assets/stacked-chart.jpeg" width="400" alt="Illustration of time spent on data preparation">

This chart is illustrative, not a universal allocation of time. Preparation effort depends on the project and data quality.

### Feature engineering techniques

 |       technique      |                                        usefulness                                |
 |:--------------------:|:--------------------------------------------------------------------------------:|
 |     `Imputation`     |                    fills out missing values in data                    |
 |   `Discretization`   |                groups a feature in some logical fashion into bins                |
 |`Categorical Encoding`|encodes categorical features into numerical values|
 |  `Feature Splitting` |splits a feature into parts|
 |   `Feature Scaling`  |handles the sensitivity of ML algorithms to the scale of input values| 
 |`Feature Expansion`|derives new features from existing ones|
 | `Log Transformation` |can reduce right skew or stabilize variance (heteroscedasticity); check the input domain       |
 |   `Outlier Handling` |investigates unusually high/low values before deciding whether to retain or treat them|
 | `RBF Transformation` |expresses similarity to reference points using radial basis functions|

### Feature engineering best practices

#### 1. **Split Dataset** into Train and Test sub-samples as early as possible

Fit learned preprocessing steps, such as imputation and scaling, on training data only. Computing their statistics before splitting lets test data influence the training process and can make evaluation overly optimistic. Fixed structural operations, such as renaming columns, can happen before splitting. Dropping rows with missing values is not automatically leakage, but it can bias the sample.

#### 2. **Feature Engineering** Includes any pre-processing techniques, such as:

- Dropping missing values
- Converting strings or non-numeric values into numeric values
- Combining features
- Creating new features

#### 3. **Feature Engineer Test Data** the same way as train data

Make sure to process the test data in the same way as training data: reuse the fitted transformers with `transform`, without refitting them. Inside cross-validation, fit preprocessing separately on each training fold using a pipeline.

Keep the target separate from the inputs, and do not impute missing target labels as if they were missing features.



<hr style="border:2px solid black">

## Example: Penguin Data

#### Load Packages

In [ ]:
# data analysis stack
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

sns.set_style("whitegrid")

#### Load Data

In [ ]:
df = pd.read_csv("./data/penguins.csv")
df.head()

#### Quick Exploration

In [ ]:
df.info()

In [ ]:
df.describe()

#### Features and Target


In [ ]:
numerical_features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm"]

categorical_features = ["species", "island", "sex"]

features = numerical_features + categorical_features

target_variable = "body_mass_g"

#### Feature-Target separation

In [ ]:
# Feature matrix
X = df[features]

# Target column
y = df[target_variable]

#### Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, shuffle=True
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

This code snippet **splits** the dataset into training and testing sets, where 25% of the data is reserved for testing. The `random_state` parameter ensures that the **split** is reproducible; the same random split will occur each time the code is run. The `shuffle` ensures that the data is shuffled before the split.


For teaching purposes, we'll demonstrate how to add a **validation set** in addition to the usual **train-test split**. Use the validation set to compare preprocessing choices and hyperparameters before final testing. The second split reserves 33% of the remaining 75%, so the overall proportions are approximately 50% training, 25% validation, and 25% test. No model is fitted in this introductory notebook.

In [ ]:
# Further split the training data into training and validation sets
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.33, random_state=42
)

print("Train shape:", X_train.shape)
print("Validation shape:", X_val.shape)

### Exploratory Data Analysis

For **Exploratory Data Analysis (EDA)**, you should concatenate `X_train` and `y_train`. This allows you to analyze also the relationship between features and the target variable.

In [ ]:
# Assuming X_train is a DataFrame and y_train is a Series
df_train = pd.concat([X_train, y_train], axis=1)

print("Combined train data shape:", df_train.shape)

**Show Some Plots**

In [ ]:
sns.pairplot(df_train, corner=True, hue="island");

The above **pairplot** shows both distribution of single variables and the relationships between two variables. 

1. **Scatter Plots**:
   - Each plot below the diagonal shows how two variables relate to each other. For example, in the plot where 'flipper_length_mm' meets 'bill_length_mm', each point represents those two measurements for one observation.

2. **Histograms/Density Plots**:
   - The plots along the diagonal show how frequently different values occur for a single variable. For instance, the plot for 'bill_depth_mm' displays the distribution of bill depths among all observations.

3. **Color**:
   - The colors represent different categories, in this case, categorized by 'island'. This helps to quickly see if measurements vary noticeably by island, with each color representing a different island.


One can create also other pairplots considering as the extra dimension `hue` the sex variable or the species variable. 

In [ ]:
df_train

>**Note** that the **indices** in the DataFrame shown above appear in a **random order**, a result of **shuffling** prior to the data split.


#### Issue with the Data

**Many models cannot handle** missing values, **categorical features** with non-numeric values, while scale-sensitive models can perform poorly with **metric features** of varying magnitudes. Proper data preprocessing is essential to prepare the data for these models.

Requirements depend on the estimator: different units are not an error in the raw data, and trees generally do not require scaling.

**Check Missing Values**

First, let's take a look at what's missing:

In [ ]:
df_train.isna().sum()

`isna()`and `isnull()` are identical methods that produce a boolean mask where True is a missing value. If we want to turn this into a useful view, we can filter using these masks.

In [ ]:
# To find the rows with NaN
df_train.loc[df_train.isna().any(axis=1)]

In [ ]:
# check missing values graphically
plt.figure(figsize=(7, 5), dpi=100)
sns.heatmap(df_train.isna());

Read the missing-value counts in the output above for `bill_length_mm`, `flipper_length_mm`, and `sex`. The counts depend on the selected training split. How do we deal with these missing values?

<hr style="border:2px solid black">

## 1. Imputation - Filling in the Blanks

#### What can we do with missing information?
There are __a few strategies__:

- __Drop__:
    + rows with missing values
    + columns with a lot of missing values
- __Fill with a value__:
    + __mean__/__median__/__mode__ of a column
    + __interpolate__ / __back fill__ / __forward fill__
    + __mean__/__median__/__mode__ of a group

- With `pandas`: 
    - `df.isna()`: checks for NaNs, then do a sum or a heatmap
    - `df.dropna()`: drop NaNs
    - `df.fillna()`: fill NaNs

Assign the result of `dropna()` or `fillna()` to retain the transformed DataFrame, or use `inplace=True` where supported to modify it directly.

Learn replacement statistics from training data only. Interpolation and forward/backward filling require a meaningful row order, such as time; do not use future observations when predicting the past. Dropping rows can also introduce selection bias.

**Missing is not zero.** In housing data, `parking_spaces = 0` means no parking; a missing value means the number is unknown. Do not replace legitimate zeros with missing markers. Whether the absence itself is informative depends on how the data was collected.

---

#### 1.1 `SimpleImputer`

We can use the scikit-learn  <a href = "https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html">`SimpleImputer()`</a>  to quickly impute the missing data in a column. There are four built-in string strategies available:

 * `strategy = 'mean'` - **the default option**, numeric only
 
 * `strategy = 'median'` - numeric only
 
* `strategy = 'most_frequent'` - mode, numeric or categorical
 
 * `strategy = 'constant'` - uses `fill_value`, numeric or categorical; if omitted, the default is 0 for numerical data or `"missing_value"` for strings/object data

If your missing values aren't NaN (i.e. - None, 0, 999, "badvalue", etc), you may need to use the `missing_values` argument to let it know what to look for.

Only treat a sentinel as missing if the data definition confirms it: a legitimate zero must not be replaced. Learn replacement values from the training data.

In [ ]:
from sklearn.impute import SimpleImputer

**Sex Column Imputation with the most frequent value**


In [ ]:
X_train["sex"].value_counts(dropna=False)

In [ ]:
# Instantiating a SimpleImputer object
sex_imputer = SimpleImputer(strategy="most_frequent")  # .set_output(transform='pandas')

In [ ]:
# Fit the variable imputer on the 'sex' column training data
sex_imputer.fit(X_train[["sex"]])

`.fit` teaches the imputer what to insert. In this example the imputer scans through the `sex` column in the training dataset (`X_train`) to determine the most frequent value in that column. After `fitting`, the imputer will internally store the mode of that column in its `statistics_` attribute.

`.transform()` reuses the stored value without learning a new replacement from validation or test data.

In [ ]:
sex_imputer.statistics_

In [ ]:
# Applying the transformation to the 'sex' column of the training data using the pre-fitted imputer.
sex_imputed_train = sex_imputer.transform(X=X_train[["sex"]])
sex_imputed_train

The `transform` method uses a pre-trained imputer to fill in the `missing values` in the `sex` column. By default, this method returns a numpy array rather than a pandas DataFrame. If you need to convert this numpy array back into a DataFrame, it's important to ensure that the index of the new DataFrame aligns with the original one to maintain data consistency.

In [ ]:
# The long way to the dataframe
sex_imputed_df_train = pd.DataFrame(
    data=sex_imputed_train,
    columns=sex_imputer.get_feature_names_out(),
    index=X_train.index,
)
sex_imputed_df_train

In [ ]:
sex_imputed_df_train.value_counts(dropna=False)

**How should we impute missing values in the test data?**

When imputing missing values - or applying other data transformations to the **test data**, it is important to **avoid using any information outside of the training data**. This helps us to avoid **data leakage** during the model building process. Therefore, you should apply only those transformations to the test data that are based on parameters established from the training data. Thus, in the context of scikit-learn, we use only the __transform method__ of the imputer or other transformation tools.

🚨🚨🚨**Very Important**🚨🚨🚨

As shown in the [machine learning workflow](machine_learning_workflow.md#the-machine-learning-workflow) reserve the test set for final evaluation after selecting the model and preprocessing. Applying an already fitted transformation is not itself leakage; using test results to choose transformations is.

The next cells demonstrate transforming held-out inputs with an already fitted imputer. This does not use their target labels or select a model. In a full project, keep test labels out of model selection and evaluate predictions only after freezing the pipeline.

In [ ]:
sex_imputed_test = sex_imputer.transform(X_test[["sex"]])
sex_imputed_test

In [ ]:
sex_imputed_df_test = pd.DataFrame(
    data=sex_imputed_test,
    columns=sex_imputer.get_feature_names_out(),
    index=X_test.index,
)
sex_imputed_df_test

---

**Imputation of 'Flipper Length' and 'Bill Depth' Columns Using Median Values**

We will follow the same steps we used for the 'sex' column to address missing values in the 'flipper length' and 'bill depth' columns. But in this case we will use the **median imputation** strategy.

In [ ]:
# Instantiating a SimpleImputer object
flipper_bill_imputer = SimpleImputer(strategy="median").set_output(transform="pandas")
flipper_bill_imputer

The `set_output(transform="pandas")` method makes supported transformers return a pandas DataFrame, preserving row indices and providing feature names.

In [ ]:
# Fit the variable imputer on the flipper_length_mm' and 'bill_depth_mm' columns of the training data
flipper_bill_imputer.fit(X_train[["flipper_length_mm", "bill_depth_mm"]])

In [ ]:
# Check the median parameters stored in the imputer after the fit
flipper_bill_imputer.statistics_

In [ ]:
X_train[["flipper_length_mm", "bill_depth_mm"]].median()

In [ ]:
# Applying the transformation to the flipper_length_mm' and 'bill_depth_mm' columns of the training data using the pre-fitted imputer.
flipper_bill_imputed_df_train = flipper_bill_imputer.transform(
    X_train[["flipper_length_mm", "bill_depth_mm"]]
)
flipper_bill_imputed_df_train.isna()

<hr style="border:2px solid black">

## 2. Categorical encoding - Replacing categories with numbers


Most algorithms aren't capable of handling strings, but there are some helpful tools to convert them into integers.


### 2.1 `OneHotEncoder` for **nominal variables** (categories without inherent order)
We can use the scikit-learn <a href = "https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html?highlight=onehotencoder#sklearn.preprocessing.OneHotEncoder">`OneHotEncoder()`</a> to get dummy values for our categorical data. It converts each category within a column into a separate binary column. Each category is represented by a **1** in its respective column for instances where it appears, and **0** where it does not. There are two options for choosing which columns to drop to avoid perfect collinearity - a common statistical issue for some models:
 
 * `drop = 'first'` - drops the first category in each feature. If there's only one category, it will drop the feature altogether.
 
 * `drop = 'if_binary'` - will only drop a category if the feature is binary (i.e. - yes/no, on/off, etc). Features with one or more than two categories will remain untouched.

After transforming our dataset, we can use `get_feature_names_out()` to get an array of feature names to label the columns.



In [ ]:
from sklearn.preprocessing import OneHotEncoder

What are the **unique categories** in the `species` column/feature?

In [ ]:
X_train["species"].unique()

In [ ]:
# Instantiating a OneHotEncoder object
species_ohencoder = OneHotEncoder(drop="first", sparse_output=True)

+ The `handle_unknown` parameter tells the encoder how to deal with categories in new data that weren't present during its initial fitting.
+ When you set `sparse_output=True`, the encoder returns the transformed data as a **sparse matrix** instead of a dense numpy array.
  + **Dense Matrix**
    + Stores all values explicitly, including zeros
    + Very memory-intensive when the matrix has a lot of zeros
  + <a href="https://en.wikipedia.org/wiki/Sparse_matrix#:~:text=In%20numerical%20analysis%20and%20scientific,of%20the%20elements%20are%20zero.">**Sparse Matrix**</a>
    + Stores only the locations and values of non-zero elements
    + Can reduce memory usage when most entries are zero
    + Can speed up suitable operations; sparse storage is not always smaller or faster for tiny or dense datasets

In [ ]:
# Fit the variable encoder on the 'species' column of the training data
species_ohencoder.fit(X=X_train[["species"]])

In [ ]:
# unique categories of the train species column learnt and stored in the encoder
species_ohencoder.categories_

In [ ]:
# new columns/feature of the categories
species_ohencoder.get_feature_names_out()

**Note that `species_Adelie` has been dropped**

In [ ]:
species_encoded_train_sparse = species_ohencoder.transform(X=X_train[["species"]])
species_encoded_train_sparse

The result is a **Compressed Sparse Row (CSR) sparse matrix**, with one row per training penguin and two columns because `Adelie` is dropped. Instead of storing every zero, CSR stores the nonzero values together with their column indices and pointers marking where each row starts. This representation is particularly useful when most entries are zero. Inspect `.shape` for the dimensions and `.nnz` for the number of stored entries (the nonzero indicators here).

In [ ]:
# Convert a sparse matrix to a dense matrix
species_encoded_train_dense = species_encoded_train_sparse.toarray()
species_encoded_train_dense

In [ ]:
# Build a labeled DataFrame while preserving the original row indices.
species_encoded_train_df = pd.DataFrame(
    data=species_encoded_train_dense,
    columns=species_ohencoder.get_feature_names_out(),
    index=X_train.index,
)
species_encoded_train_df

In [ ]:
# Let's see the encoded species columns for two penguins
species_encoded_train_df.loc[[24, 323]]

**How to read the 👆🏽 DataFrame**?

+ Penguin 24 has both indicators equal to 0, so its species is the reference category `Adelie`
+ Penguin 323 has `species_Gentoo = 1`, so it belongs to `Gentoo` 

With `drop="first"`, both indicators being zero identifies the reference category `Adelie`. This encoder raises an error for unseen species: using `handle_unknown="ignore"` here would make an unseen species indistinguishable from the dropped reference category.

### 2.2 `OrdinalEncoder` for **Ordinal Variables** (Categories with Inherent Order)


We can use the scikit-learn <a href="https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OrdinalEncoder.html">`OrdinalEncoder()`</a> to encode ordinal categories into meaningful numeric values. This encoder is particularly useful for categories that have a natural ordered relationship. Here are some key points:

* `categories` - specifies the order of categories explicitly if the default lexical order is not desired.

* `dtype` - the data type of the output (default is `float64`).

* `handle_unknown` - decides how to handle unknown categories that appear during transformation:
  
  * `handle_unknown='error'` - **the default option**, throws an error if an unknown category is encountered.
  
  * `handle_unknown='use_encoded_value'` - allows assigning a specific integer for unknown categories with additional arg `unknown_value`.

If your categories have a meaningful order, specify this in the `categories` argument to ensure the encoding respects the ordinal nature.


As in the penguins dataset we don't have a feature with ordinal categories, let's consider a simple case involving an ordinal feature: "Education Level". The categories, listed in order of their educational achievement, are:

+ High School
+ Bachelor's
+ Master's
+ Doctorate

In ordinal encoding, each category is assigned a unique integer based on its order.
Here is how you can map these categories:
+ High School --> 0
+ Bachelor's --> 1
+ Master's  --> 2
+ Doctorate --> 3
  


In [ ]:
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
train_data = [
    ["High School"],
    ["Bachelor's"],
    ["Doctorate"],
    ["Master's"],
    ["Doctorate"],
]
df_train_data_ = pd.DataFrame(data=train_data, columns=pd.Index(["education_level"]))
df_train_data_

In [ ]:
# Instantiate the encoder object
edu_encoder = OrdinalEncoder(
    categories=[["High School", "Bachelor's", "Master's", "Doctorate"]], dtype=int
).set_output(transform="pandas")

In [ ]:
# Fit the variable encoder on the 'education_level' column of the training data
edu_encoder.fit(df_train_data_[["education_level"]])

In [ ]:
# learned and stored categories by the encoder
edu_encoder.categories_

In [ ]:
# Encode the 'education_level' column in the train data
edu_encoded_df_train = edu_encoder.transform(df_train_data_)
edu_encoded_df_train

#### Summary

`OrdinalEncoder()` and `OneHotEncoder()` turn categorical data into numerical values

`OrdinalEncoder()` results in one column per input feature

`OneHotEncoder()` creates category-indicator columns for each input feature

`drop="first"` in scikit-learn (`drop_first=True` in pandas) removes a column from your dummy frame to avoid **perfect collinearity**, especially when using a regression model. There is some discussion of why multicollinearity is <a href = "https://towardsdatascience.com/multicollinearity-why-is-it-bad-5335030651bf">a problem</a> and <a href = "https://stats.stackexchange.com/questions/224051/one-hot-vs-dummy-encoding-in-scikit-learn">why it might not be that bad</a>. Dropping a reference category avoids redundant columns in an unregularized linear model with an intercept. It is not a universal rule: regularized models can retain all categories.

Ordinal codes impose numerical gaps as well as an order; those gaps are not automatically meaningful for linear or distance-based models.


- `LabelEncoder` encodes target labels `y`, not input features `X`; it is similar to [factorize()](https://pandas.pydata.org/docs/reference/api/pandas.factorize.html) in Pandas 
- `OneHotEncoder` similar to [get_dummies()](https://pandas.pydata.org//docs/reference/api/pandas.get_dummies.html) in Pandas

Unlike calling `get_dummies` separately on each dataset, a fitted `OneHotEncoder` reuses the training category schema on new data. Choose how to handle unseen categories explicitly.

<hr style="border:2px solid black">

## 3. Discretization (Binning) - Splitting scalars into categories
Breaking a continuous variable into buckets can lead to some effects:
+ it can reduce the model's sensitivity to minor fluctuations and noise in the data, reducing the risk of overfitting
+ it can bring loss of important details in the data if the bin widths are not properly chosen, leading to higher risk of underfitting
+ one-hot-encoded bins let a linear model learn a piecewise-constant relationship with the original numerical feature:
  + [regression example from scikit-learn](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_discretization.html)
  + [classification example from scikit-learn](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_discretization_classification.html#sphx-glr-auto-examples-preprocessing-plot-discretization-classification-py)

### `KBinsDiscretizer()`

We can use <a href = "https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.KBinsDiscretizer.html">`KBinsDiscretizer()`</a> to turn a set of scalars into bins, and then one-hot encode these bins in a single step. A few parameters to keep in mind:
 
* `n_bins` - choose how many bins to generate, default = 5.
 
* `strategy = 'quantile'` - **default option**, generate bins of equal population.
 
* `strategy = 'uniform'` - generate bins of equal width.
  
* `encode = 'onehot' ` - **default option**, encode the transformed result with one-hot encoding and return a sparse matrix.

In [ ]:
sns.displot(data=X_train, x="bill_length_mm", bins=5);

In [ ]:
from sklearn.preprocessing import KBinsDiscretizer

In [ ]:
# Instantiate  KBinsDiscretizer object
bill_length_binner = KBinsDiscretizer(
    n_bins=5, encode="onehot-dense", strategy="quantile", quantile_method="linear"
).set_output(transform="pandas")

In [ ]:
# Fit the variable binner on the 'bill_length_mm' column of the training data
bill_length_binner.fit(X_train[["bill_length_mm"]])

In [ ]:
# Access and print the bin edges for the 'bill_length_mm' feature
bin_edges = bill_length_binner.bin_edges_
print("Bin edges for 'bill_length_mm':", bin_edges[0])

The bin edges represent the `quintiles`. They divide the data into 5 more-or-less equal parts by count, not necessarily by width. Tied values can make the counts unequal.

In [ ]:
# Get the quintiles
X_train[["bill_length_mm"]].quantile(q=[0.0, 0.2, 0.4, 0.6, 0.8, 1.0]).rename(
    {"bill_length_mm": "bill_quintiles"}, axis=1
)

In [ ]:
# Plotting the bill length distribution using the learnt bin edges
sns.displot(data=X_train, x="bill_length_mm", bins=bin_edges[0]);

With quantile bins, the counts are approximately equal, although the bin widths can differ. In a density-normalized histogram, equal counts correspond to equal areas, not necessarily equal heights.

In [ ]:
# Transform 'bill_length_mm' using the quintile bins defined in 'bill_length_binner'
bill_length_binned_df_train = bill_length_binner.transform(X_train[["bill_length_mm"]])
bill_length_binned_df_train

In [ ]:
pd.DataFrame(
    {
        "bin": range(len(bin_edges[0]) - 1),
        "lower_edge_mm": bin_edges[0][:-1],
        "upper_edge_mm": bin_edges[0][1:],
    }
)

**How to read the transformed DataFrame?**

Each penguin belongs to one of five bins (0 through 4). The table above shows their actual boundaries learned from the training data. An indicator of 1 identifies the row's bin; the other indicators are 0.

Internal boundaries belong to the bin on their right; values beyond the training range go into the first or last bin. Compare each transformed row with its original bill length.

In [ ]:
# Bill_length value for the observation 24
X_train[["bill_length_mm"]].loc[24]

- `KBinsDiscretizer(strategy='quantile')`  similar to [qcut()](https://pandas.pydata.org/docs/reference/api/pandas.qcut.html) in Pandas 
- `KBinsDiscretizer(strategy='uniform')` similar to [cut()](https://pandas.pydata.org/docs/reference/api/pandas.cut.html) in Pandas

Unlike recomputing bins on each dataset, the fitted discretizer reuses training boundaries when transforming validation or test data.

<hr style="border:2px solid black">

## 4. Numerical Feature Scaling - Making Scales Comparable

The goal of feature scaling is to transform numerical features to be on a similar scale. Here, scaling operates column by column; it is different from row normalization, which rescales each observation to unit norm.

For example, consider the following two features:
+ `income` spans from 20_000 to 100_000 euro:
+ `age` spans from 20 to 100 years
  
Without scaling, the income feature would disproportionately influence [any distance-based algorithms](https://towardsdatascience.com/all-about-feature-scaling-bcc0ad75cb35/) because of its larger range of values compared to age.

By applying feature scaling, both income and age can be put on comparable numerical scales. This prevents units alone from dominating distances, but does **not** guarantee equal feature importance or better model performance. Compare the alternatives using validation data.


### 4.1 Min-Max Scaling

It transforms the training values of a nonconstant numerical feature to a fixed range, typically between 0 and 1, by subtracting the minimum value and dividing by the range as shown in the formula below:

$$\large X_{scaled} = \large \dfrac{X - X_{min}}{X_{max} - X_{min}}$$

$X_{min}$ and $X_{max}$ are learned from **training data** and reused on new data.

After transformation:
+ Training features are constrained to a specific range (e.g., 0 to 1)
  + Minimum value becomes 0, maximum value becomes 1.
+ The ordering and distribution shape within each feature are preserved.
  + Everything in between 0 and 1 is proportionally distributed.
+ Outliers are not handled well: extreme values can compress the other observations into a narrow range.

New values may fall outside [0, 1]. Constant training features are handled separately and map to 0 with the default settings.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# Instantiate the scaler
min_max_scaler = MinMaxScaler(feature_range=(0, 1)).set_output(transform="pandas")

In [ ]:
# Fit the scaler to the columns 'bill_length_mm' and 'flipper_length_mm' of the training data
min_max_scaler.fit(X=X_train[["bill_length_mm", "flipper_length_mm"]])

In [ ]:
# Learnt Maximum values for each feature on the train data
print("Max values:", min_max_scaler.data_max_)

# Learnt Minimum values for each feature on the train data
print("Min values:", min_max_scaler.data_min_)

# Learnt Range for each feature on the train data
print("Range:", min_max_scaler.data_range_)

In [ ]:
X_train[["bill_length_mm", "flipper_length_mm"]].max()

In [ ]:
# Scale the features
min_max_scaled_bill_fli_df_train = min_max_scaler.transform(
    X=X_train[["bill_length_mm", "flipper_length_mm"]]
)
min_max_scaled_bill_fli_df_train

In [ ]:
min_max_scaled_bill_fli_df_train.describe()

**Note** that the **min** and **max** value for each features are respectively 0 and 1

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=2)

# Plotting unscaled data
sns.histplot(data=X_train, x="bill_length_mm", ax=ax[0, 0], kde=True)
ax[0, 0].set_title("Unscaled Bill Length")

sns.histplot(data=X_train, x="flipper_length_mm", ax=ax[0, 1], kde=True)
ax[0, 1].set_title("Unscaled Flipper Length")

# Plotting scaled data
sns.histplot(
    data=min_max_scaled_bill_fli_df_train, x="bill_length_mm", ax=ax[1, 0], kde=True
)
ax[1, 0].set_title("Scaled Bill Length")

sns.histplot(
    data=min_max_scaled_bill_fli_df_train, x="flipper_length_mm", ax=ax[1, 1], kde=True
)
ax[1, 1].set_title("Scaled Flipper Length")

ax[1, 0].set_xlabel("Bill length (scaled, unitless)")
ax[1, 1].set_xlabel("Flipper length (scaled, unitless)")

# Adjust layout
plt.tight_layout()

As shown in the visualization above. The underlying pattern of the data remains preserved after min-max scaling. 

### 4.2 Standard Scaling

It transforms the training values of a nonconstant numerical feature so that they have a mean of zero and a standard deviation of one. This is accomplished by subtracting the mean of the feature from each value and then dividing by the standard deviation, as shown in the formula below:

$$\large X_{\text{standardized}} = \frac{X - \mu}{\sigma}$$

Where:
- $\mu$ is the mean of the training feature values.
- $\sigma$ is the population standard deviation of the training feature values.

**After transformation:**
+ **Centered Data**: The mean of the transformed training data is 0.
+ **Unit Variance**: The standard deviation becomes 1, which means that feature variance is normalized. Constant features instead become 0 after centering, with variance 0.
+ **Preserves Relationships**: The ordering and distribution shape within each feature are maintained, similar to Min-Max scaling. Standardization does not make a distribution Gaussian.
+ **Handling Outliers**: Mean and standard deviation are sensitive to outliers, so standard scaling is not a robust outlier treatment.

Reuse the training mean and standard deviation on new data; their transformed mean and standard deviation need not be 0 and 1.

 **[`StandardScaler()`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html)**

### 4.3 Robust Scaling

It transforms the values of a numerical feature by subtracting the median and then dividing by the interquartile range (IQR), as shown in the formula below (for a nonzero IQR):

$$\large X_{\text{robust}} = \frac{X - \text{median}(X)}{\text{IQR}}$$

Where:
- **median(X)** is the median of the training feature values.
- **IQR** is the interquartile range, which is the difference between the 75th percentile (Q3) and the 25th percentile (Q1) of the training feature values.

After transformation:
+ **Centered and Scaled Data**: With the default settings, the median of the transformed training data becomes 0 and its IQR becomes 1, not necessarily its variance. A zero IQR is handled separately to avoid division by zero.
+ **Handling Outliers**: Since the median and IQR are less affected by outliers than the mean and standard deviation, this scaling method can be better suited for datasets with outliers. It does not remove outliers or guarantee better model performance.
+ **Preserving Relationships**: Like other scaling methods, robust scaling maintains the ordering and distribution shape within each feature, including its outliers.

Reuse the training median and IQR on new data; their transformed median and IQR need not be 0 and 1.

**[`RobustScaler()`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.RobustScaler.html)**

#### Compare the Three Scalers

The same small sample contains ordinary values and two extreme values. Compare the spacing of the ordinary observations, not just the axis limits. Robust scaling keeps the extremes: it does not delete or cap them.

In [ ]:
from sklearn.preprocessing import RobustScaler, StandardScaler

sample = np.array([8, 9, 10, 11, 12, 13, 14, 45, 60]).reshape(-1, 1)
scalers = {
    "Original": None,
    "MinMax": MinMaxScaler(),
    "Standard": StandardScaler(),
    "Robust": RobustScaler(),
}
scaled_values = {}
fig, axes = plt.subplots(4, 1, figsize=(8, 5), constrained_layout=True)
for ax, (name, scaler) in zip(axes, scalers.items()):
    values = sample.ravel() if scaler is None else scaler.fit_transform(sample).ravel()
    scaled_values[name] = values
    # Color the same two extreme observations in every panel.
    ax.scatter(values[:-2], np.zeros(7), label="Ordinary values")
    ax.scatter(values[-2:], np.zeros(2), color="crimson", label="Extreme values")
    ax.set(title=name, yticks=[], xlabel="Value" if scaler is None else "Scaled value")
axes[0].legend(loc="upper center", ncol=2)
plt.show()
# Compare numerical values directly: each plot above uses its own axis limits.
pd.DataFrame(scaled_values).round(3)

All three scalers preserve ordering. MinMax uses the extremes to define its range; Standard uses the mean and standard deviation; Robust uses the median and IQR. The panels look alike because these are affine transformations and each axis is adjusted independently. Use the table to compare the actual scales: the extreme values remain extreme, even after robust scaling. None guarantees better predictions: the model and validation results determine what helps.

### 4.4 Log Scaling
Log scaling is a non-linear transformation that uses the logarithmic function to reduce the range and variation of data values.

**Log transformation** applies a logarithmic scale to the values of a numerical feature, typically using the natural logarithm (base e), although any logarithm base can be used depending on the data and the specific needs. The transformation is given by the formula:

$$\large X_{\text{log}} = \log(X)$$

Where:
- **log()** denotes the logarithmic function, which could be to any base.

**After transformation:**
+ **Reduces Scale Differentials**: the scale of high magnitude values is more significantly compressed than those of low magnitude.
+ **Handling Skewness**: Can reduce right skew, but does not guarantee a uniform or Gaussian distribution.
+ **Stabilizing Variance**: Compressing larger values can help when variability increases with the mean, but constant variance is not guaranteed.
+ **Reducing Impact of Outliers**: Outliers that are far from the majority of data points become less dominating after log transformation due to the compression effect at higher value ranges.

**Considerations:**
- Log transformation can only be applied to positive values. For nonnegative values including zero, `log1p(X)` computes `log(1 + X)`. Adding a constant is only appropriate when justified by the domain: choose it using training data, reuse it, and check that future inputs remain valid. For signed data, consider a transformation such as Yeo-Johnson.



While **scikit-learn** itself does not provide a direct transformer for log transformations, you can easily implement log transformation using a custom transformer with the help of [**FunctionTransformer**](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.FunctionTransformer.html). This allows you to apply any function, including a logarithmic transformation, to your data.


In [ ]:
from sklearn.preprocessing import FunctionTransformer

In [ ]:
# Define the log transformation function
# c=1 allows zero for nonnegative inputs; always require X + c > 0.
# This fixed shift is an illustration, not a guarantee of better features.
def log_transformation(X, c):
    return np.log(X + c)


def inverse_log_transformation(Xlog, c):
    return np.exp(Xlog) - c

In [ ]:
# Create the transformer using the log_transform function
log_transformer = FunctionTransformer(
    func=log_transformation,
    inverse_func=inverse_log_transformation,
    kw_args={"c": 1},
    inv_kw_args={"c": 1},
    feature_names_out="one-to-one",
).set_output(transform="pandas")

In [ ]:
# Fit the transformer to the 'flipper_length_mm' column of X_train
# Check the inverse on observed values; NaNs are not comparable.
# This transformer learns no statistics, and transform preserves missing rows.
log_transformer.fit(X=X_train[["flipper_length_mm"]].dropna())

In [ ]:
# Transform the data using the fitted transformer
log_transformed_flip_df_train = log_transformer.transform(
    X_train[["flipper_length_mm"]]
)
log_transformed_flip_df_train

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=1)

# Plotting unscaled data
sns.histplot(data=X_train, x="flipper_length_mm", ax=ax[0], kde=True)
ax[0].set_title("Unscaled Flipper Length")

sns.histplot(
    data=log_transformed_flip_df_train, x="flipper_length_mm", ax=ax[1], kde=True
)
ax[1].set_title("Scaled Flipper Length")

# Adjust layout
plt.tight_layout()

#### Readings on Scaling
+ [Compare the effect of different scalers on data with outliers](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_all_scaling.html#sphx-glr-auto-examples-preprocessing-plot-all-scaling-py)
+ [Importance of feature scaling](https://scikit-learn.org/stable/auto_examples/preprocessing/plot_scaling_importance.html)


<hr style="border:2px solid black">

## 5. Feature Expansion
### `PolynomialFeatures`
We can use the scikit-learn [`PolynomialFeatures()`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.PolynomialFeatures.html) to add polynomial or interaction features to our dataset. This transformer lets even a linear model capture nonlinear relationships in the original inputs while remaining linear in its coefficients. Here are some key points:

* `degree` - specifies the degree of the polynomial features (default is 2). For example, for a single feature \(X\), if `degree=2`, it generates \(X, X^2\), plus a constant column when `include_bias=True`.

* `interaction_only` - if set to `True`, this will produce features that are the product of distinct input features. For example, if two features are \(X_1\) and \(X_2\), it will generate \(X_1 \times X_2\) but not \(X_1^2\) or \(X_2^2\).

* `include_bias` - decides whether to include a bias column (the feature column consisting of ones). This can be set to `False` if a bias is already handled or not required in the model.

🚨**Very important**🚨:
+ **there is no universal rule to scale only before expansion**. Scaling first can limit numerical growth; scaling after expansion can help regularized models because the expanded columns have different scales. Fit learned transformations on training data only.

### 5.1 `Polynomial Terms`

- Additional features obtained by an existing feature to some power
- Non-linear relationships can be modelled
- For some feature x, consider the model: 

$$
y = a_0 + a_1x + a_2x^2 +\ldots+\epsilon
$$

- Potential improvement of model accuracy, but increased risk of overfitting

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

In [ ]:
# Instantiate the polynomial features object
poly_expansion = PolynomialFeatures(
    degree=2, interaction_only=False, include_bias=False
).set_output(transform="pandas")

In [ ]:
poly_expansion.fit(min_max_scaled_bill_fli_df_train[["bill_length_mm"]])

In [ ]:
expanded_scaled_bill_length_df_train = poly_expansion.transform(
    min_max_scaled_bill_fli_df_train[["bill_length_mm"]]
)
expanded_scaled_bill_length_df_train

### 5.2 `Interaction Terms`

- For multiple initial features, there could be *interactions* (cross-polynomial terms)
- For 2 features, $x_0$ and $x_1$ for example, a 2nd-degree polynomial may contain:

$$
1,~x_0,~x_1,~x_0^2,~x_0x_1,~x_1^2
$$

- Each of the terms gets their own coefficient in a regression model
- The example below uses `interaction_only=False` to show both powers and cross-products. With `interaction_only=True`, squared terms disappear but the original features remain. `include_bias=False` omits the constant column shown in the general formula.

In [ ]:
# Instantiate the polynomial features object
poly_expansion = PolynomialFeatures(
    degree=2, interaction_only=False, include_bias=False
).set_output(transform="pandas")

Since PolynomialFeatures doesn’t support NaNs, we must preprocess the data before calling .fit()

Let's check for missing values first.

In [ ]:
np.isnan(min_max_scaled_bill_fli_df_train).sum()

Polynomial expansion does not accept missing inputs. Impute the remaining missing values rather than dropping rows independently of `y_train`, which could misalign features and target.

In [ ]:
# Impute missing values without removing rows from the feature matrix.
expansion_imputer = SimpleImputer(strategy="median").set_output(transform="pandas")
min_max_scaled_bill_fli_df_train = expansion_imputer.fit_transform(
    min_max_scaled_bill_fli_df_train
)

In [ ]:
# check for missings
np.isnan(min_max_scaled_bill_fli_df_train).sum()

In [ ]:
poly_expansion.fit(X=min_max_scaled_bill_fli_df_train)

In [ ]:
expanded_scaled_bill_fli_length_df_train = poly_expansion.transform(
    min_max_scaled_bill_fli_df_train
)
expanded_scaled_bill_fli_length_df_train

**Note** that the column `bill_length_mm flipper_length_mm` represents the interaction term. If `interaction_only = True` the quadratic features `bill_length_mm^2` and `flipper_length_mm^2` will not be present.

<hr style="border:2px solid black">

## Optional: Advanced Imputation

The core lesson is complete. Group-wise, iterative, and KNN imputation are optional extensions, not prerequisites for the exercises. You can skip to the transformation summary at the end.

### Beyond Simple Imputation

Previously, we looked at basic imputation methods that use simple statistics like the **mean** or **median** from the same column with missing values.
As always, there is more to discover. There are also more advanced imputation techniques that consider relationships between different features.

#### Group-wise Mean/Median/Mode Imputation:
Imputes missing values based on the mean, median, or mode calculated within subgroups of the data.

The example below learns group means from training data. For unseen or missing groups, or groups with no observed value, it uses the overall training mean. Grouping variables must be available when predicting. A feature with no observed training values raises an error rather than silently remaining missing.

To build a group-wise imputer that follows the scikit-learn interface of `.fit()` and `.transform()`, we will create a custom class based  on scikit-learn's `BaseEstimator` and `TransformerMixin` classes

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.utils.validation import check_is_fitted

In [ ]:
class GroupMeanImputer(TransformerMixin, BaseEstimator):
    def __init__(self, group_column, target_columns):
        # Identify the grouping column and the columns to impute.
        self.group_column = group_column
        self.target_columns = target_columns

    def fit(self, X, y=None):
        # Learn group means and a fallback from training data only.
        self.means_ = X.groupby(self.group_column)[self.target_columns].mean()
        self.fallback_ = X[self.target_columns].mean()
        if self.fallback_.isna().any():
            raise ValueError("Each imputed feature needs an observed training value.")
        return self

    def transform(self, X):
        check_is_fitted(self, ["means_", "fallback_"])
        # Preserve the row index and leave the original input unchanged.
        result = X[self.target_columns].copy()
        for column in self.target_columns:
            group_means = X[self.group_column].map(self.means_[column])
            # Unseen, missing, or entirely empty groups use the training mean.
            fill_values = group_means.fillna(self.fallback_[column])
            result[column] = result[column].fillna(fill_values)
        return result

In [ ]:
# Instantiate the group-wise imputer
group_mean_imputer = GroupMeanImputer(
    group_column="island", target_columns=["bill_depth_mm", "flipper_length_mm"]
)

In [ ]:
# Fit the imputer
group_mean_imputer.fit(X=X_train)

In [ ]:
group_mean_imputer.means_

In [ ]:
group_mean_imputer.means_.index

In [ ]:
group_mean_imputer.transform(X=X_train)

### Model Imputation:
+ Use a model to predict the missing values based on other variables
+ For iterative imputation in scikit-learn there is  [IterativeImputer()](https://scikit-learn.org/stable/auto_examples/impute/plot_iterative_imputer_variants_comparison.html#sphx-glr-auto-examples-impute-plot-iterative-imputer-variants-comparison-py)

Fit the imputer on training data only. A more complex imputation model does not automatically produce better estimates.

### K-Nearest Neighbors (KNN) Imputation:
+ Imputes missing entries based on the k-nearest neighbors found by measuring distance from other points.
+ In scikit-learn there is [KNNImputer()](https://scikit-learn.org/stable/modules/generated/sklearn.impute.KNNImputer.html)

## Choosing a Transformation

| Input / issue | Starting option | Main precaution |
|---|---|---|
| Missing numerical values | Median imputation | Learn the median on training data; inspect why values are missing. |
| Missing categories | Most frequent value or a missing category | Imputation can hide meaningful missingness. |
| Nominal categories | One-hot encoding | Decide how to handle unseen categories; watch column count. |
| Ordered categories | Explicit ordinal encoding | Numeric gaps do not necessarily represent real distances. |
| Different numerical units | Standard, MinMax, or Robust scaling | Model-dependent; scaling does not remove outliers. |
| Right-skewed positive values | Log transformation | Check the domain and validate its benefit. |
| Nonlinear relationships | Bins or polynomial features | More features can overfit; compare on validation data. |

These are starting points, not rules. Learn preprocessing inside each training fold when comparing pipelines.